# Amazon Search / LTR — Week 4 V3 One-Click Colab
## Coverage-Aware Hybrid Candidate Retrieval
### BM25 + SBERT + CLIP, without entering Week 5 Learning-to-Rank

This notebook is the **next Week 4 step** after the naive SBERT+CLIP score fusion experiment.

### Why V3 exists
Week 4 V2 showed:
- SBERT improved NDCG/MRR but lost some Recall@50 vs BM25.
- CLIP provided useful visual signals for some queries.
- Naive `0.7 * SBERT + 0.3 * CLIP` hurt Recall@50 because only about half of the catalog had usable images.

### V3 goal
Use CLIP as an **additional candidate source**, not as a penalty for products that have no image.

### This notebook tests four Week-4-only retrieval architectures
1. **BM25 + SBERT RRF**
2. **SBERT + CLIP RRF**
3. **BM25 + SBERT + CLIP RRF**
4. **Coverage-aware score fusion**
   - BM25 + SBERT for no-image products
   - BM25 + SBERT + CLIP when an image exists
   - weights are renormalized per product so missing images are not penalized

### Evaluation
- Judged-candidate NDCG@10
- Judged-candidate MRR
- Full-catalog known-relevant Recall@10
- Full-catalog known-relevant Recall@50
- Candidate-union Recall@100 / Recall@300
- Retrieval latency

### Development gate
Roadmap target:
`multimodal Recall@50 >= 10% above SBERT text-only Recall@50`

No LambdaMART, LightGBM, learned ranking model, or Week 5 feature engineering is used here.

In [ ]:
# ============================================================
# 0. CONFIG
# ============================================================

TOP_K = 50
CANDIDATE_DEPTH = 100
RRF_K = 60

# Fixed weights declared BEFORE evaluation.
# CLIP gets a smaller weight because image coverage is partial.
RRF_W_BM25 = 1.00
RRF_W_SBERT = 1.00
RRF_W_CLIP = 0.35

# Coverage-aware continuous score fusion.
SCORE_W_BM25 = 0.35
SCORE_W_SBERT = 0.50
SCORE_W_CLIP = 0.15

SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SBERT_BATCH_SIZE = 128
KNOWN_RELEVANT_THRESHOLD = 2
LATENCY_SAMPLE_SIZE = 30
RANDOM_SEED = 42

print({
    "TOP_K": TOP_K,
    "CANDIDATE_DEPTH": CANDIDATE_DEPTH,
    "RRF_K": RRF_K,
    "RRF weights": [RRF_W_BM25, RRF_W_SBERT, RRF_W_CLIP],
    "score weights": [SCORE_W_BM25, SCORE_W_SBERT, SCORE_W_CLIP],
})

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys, subprocess

packages = [
    "sentence-transformers>=3.0,<6",
    "faiss-cpu>=1.8",
    "rank-bm25>=0.2.2",
    "pyarrow>=15",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed.")

In [ ]:
# ============================================================
# 2. IMPORTS + DIRECTORIES
# ============================================================

import os, re, sys, json, time, shutil, random, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ndcg_score
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
INPUT_DIR = ROOT / "week4_v3_inputs"
EXTRACT_DIR = ROOT / "week4_v3_extracted"
OUTPUT_DIR = ROOT / "week4_v3_outputs"

for p in [INPUT_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 3. UPLOAD THREE INPUT ZIP FILES
# ============================================================

# Upload:
# 1) amazon_week1_v4_processed_data.zip
# 2) amazon_week4_v1_sbert_faiss_results.zip
# 3) amazon_week4_v2_clip_multimodal_results.zip

is_colab = False
try:
    from google.colab import files
    is_colab = True
except Exception:
    files = None

if is_colab:
    print("Upload THREE ZIP files:")
    print("1) Week 1 processed-data ZIP")
    print("2) Week 4 V1 SBERT/FAISS results ZIP")
    print("3) Week 4 V2 CLIP/multimodal results ZIP")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        with open(INPUT_DIR / filename, "wb") as f:
            f.write(content)
else:
    print("Place the three ZIPs in:", INPUT_DIR)

zip_files = sorted(INPUT_DIR.glob("*.zip"))

if len(zip_files) < 3:
    for z in ROOT.glob("*.zip"):
        if z.parent != INPUT_DIR:
            dst = INPUT_DIR / z.name
            if not dst.exists():
                shutil.copy2(z, dst)
    zip_files = sorted(INPUT_DIR.glob("*.zip"))

print("\nZIPs found:")
for z in zip_files:
    print(" -", z.name)

if len(zip_files) < 3:
    raise FileNotFoundError("Need Week 1 + Week 4 V1 + Week 4 V2 ZIP files.")

In [ ]:
# ============================================================
# 4. EXTRACT + AUTO-DETECT FILES
# ============================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for zpath in zip_files:
    out = EXTRACT_DIR / zpath.stem
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(out)
    print("Extracted:", zpath.name)

def find_one(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing required file: {filename}")
    return matches[0]

PRODUCTS_PATH = find_one("cleaned_products_10k.parquet")
QPR_PATH = find_one("query_product_relevance.parquet")

SBERT_PRODUCT_EMB_PATH = find_one("week4_sbert_product_embeddings.npy")
TEST_QUERIES_PATH = find_one("week4_test_queries_used.csv")
V1_METRICS_PATH = find_one("week4_text_retrieval_metrics.csv")

CLIP_IMAGE_EMB_PATH = find_one("week4_clip_image_embeddings.npy")
CLIP_QUERY_EMB_PATH = find_one("week4_clip_query_embeddings.npy")
CLIP_MAP_PATH = find_one("week4_clip_embedding_map.csv")
V2_METRICS_PATH = find_one("week4_v2_final_metrics.csv")
IMAGE_COVERAGE_PATH = find_one("week4_v2_image_coverage.csv")

print("All required artifacts detected.")

In [ ]:
# ============================================================
# 5. LOAD + ALIGN DATA
# ============================================================

products = pd.read_parquet(PRODUCTS_PATH)
qpr = pd.read_parquet(QPR_PATH)
test_queries = pd.read_csv(TEST_QUERIES_PATH)

products["product_id"] = products["product_id"].astype(str)
qpr["product_id"] = qpr["product_id"].astype(str)
qpr["query_id"] = qpr["query_id"].astype(str)
test_queries["query_id"] = test_queries["query_id"].astype(str)

products = products.drop_duplicates("product_id").reset_index(drop=True)

sbert_product_embeddings = np.load(SBERT_PRODUCT_EMB_PATH).astype("float32")
clip_image_embeddings = np.load(CLIP_IMAGE_EMB_PATH).astype("float32")
clip_query_embeddings = np.load(CLIP_QUERY_EMB_PATH).astype("float32")
clip_map = pd.read_csv(CLIP_MAP_PATH)

if len(products) != len(sbert_product_embeddings):
    raise ValueError("SBERT embedding rows do not match catalog rows.")
if len(test_queries) != len(clip_query_embeddings):
    raise ValueError("CLIP query embeddings do not match V1 test queries.")

product_ids = products["product_id"].tolist()
idx_to_pid = np.array(product_ids, dtype=object)
pid_to_idx = {pid:i for i,pid in enumerate(product_ids)}

clip_map["product_id"] = clip_map["product_id"].astype(str)
clip_catalog_indices = clip_map["catalog_index"].astype(int).to_numpy()

# Verify saved V2 catalog mapping.
mapped_ids = idx_to_pid[clip_catalog_indices]
if not np.all(mapped_ids == clip_map["product_id"].to_numpy()):
    raise ValueError("CLIP embedding map does not align with Week 1 product order.")

gold = qpr[qpr["query_id"].isin(test_queries["query_id"])].copy()

print("Catalog products:", len(products))
print("Queries:", len(test_queries))
print("CLIP-covered products:", len(clip_map))
print("CLIP coverage:", round(len(clip_map)/len(products)*100, 2), "%")

In [ ]:
# ============================================================
# 6. BUILD PRODUCT TEXT + BM25
# ============================================================

def safe_col(df, col):
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)

existing_text = safe_col(products, "product_text").str.strip()

reconstructed = (
    safe_col(products, "title").str.strip() + " " +
    safe_col(products, "brand").str.strip() + " " +
    safe_col(products, "description").str.strip() + " " +
    safe_col(products, "features").str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

products["retrieval_text"] = np.where(
    existing_text.str.len() > 0,
    existing_text,
    reconstructed
)

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[._/-][a-z0-9]+)*", re.I)

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

print("Tokenizing catalog...")
tokenized_corpus = [tokenize(x) for x in products["retrieval_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 ready.")

In [ ]:
# ============================================================
# 7. RECOMPUTE SBERT QUERY EMBEDDINGS
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

sbert = SentenceTransformer(SBERT_MODEL, device=device)

queries = test_queries["query"].fillna("").astype(str).tolist()

sbert_query_embeddings = sbert.encode(
    queries,
    batch_size=SBERT_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("SBERT query embedding shape:", sbert_query_embeddings.shape)

In [ ]:
# ============================================================
# 8. EVALUATION HELPERS
# ============================================================

relevance_lookup = defaultdict(dict)
for row in gold[["query_id","product_id","relevance_label"]].itertuples(index=False):
    relevance_lookup[row.query_id][row.product_id] = int(row.relevance_label)

known_positives = (
    gold[gold["relevance_label"] >= KNOWN_RELEVANT_THRESHOLD]
    .groupby("query_id")["product_id"]
    .apply(lambda x:set(x.astype(str)))
    .to_dict()
)

def recall_at_k(retrieved_ids, positives, k):
    positives = set(positives)
    if not positives:
        return np.nan
    return len(set(retrieved_ids[:k]) & positives) / len(positives)

def judged_metrics(qid, score_by_pid, k=10):
    rows = gold[gold["query_id"] == qid][["product_id","relevance_label"]].copy()
    rows["score"] = rows["product_id"].map(score_by_pid).fillna(-1e9)

    if len(rows) == 0:
        return np.nan, np.nan

    labels = rows["relevance_label"].astype(float).to_numpy()
    scores = rows["score"].astype(float).to_numpy()

    ndcg = float(ndcg_score([labels], [scores], k=min(k, len(rows))))

    ranked = rows.sort_values("score", ascending=False)
    mrr = 0.0
    for rank, rel in enumerate(ranked["relevance_label"].tolist(), start=1):
        if rel > 0:
            mrr = 1.0/rank
            break

    return ndcg, mrr

def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = float(np.min(x)), float(np.max(x))
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x-lo)/(hi-lo)

def top_indices(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k-1)[:k]
    return idx[np.argsort(-scores[idx])]

def rank_map(indices):
    return {int(idx): rank for rank, idx in enumerate(indices, start=1)}

def summarize(df):
    return {
        "NDCG@10_judged": df["NDCG@10_judged"].mean(),
        "MRR_judged": df["MRR_judged"].mean(),
        "Recall@10_full": df["Recall@10_full"].mean(),
        "Recall@50_full": df["Recall@50_full"].mean(),
        "CandidateRecall@100": df["CandidateRecall@100"].mean(),
        "CandidateRecall@300": df["CandidateRecall@300"].mean(),
        "queries_evaluated": len(df),
    }

In [ ]:
# ============================================================
# 9. RUN V3 HYBRID RETRIEVAL
# ============================================================

methods = [
    "BM25+SBERT_RRF",
    "SBERT+CLIP_RRF",
    "BM25+SBERT+CLIP_RRF",
    "CoverageAware_ScoreFusion"
]

top50_rows = []
metric_rows = []
latency_rows = []

for qi, row in enumerate(test_queries.itertuples(index=False)):
    qid, query = row.query_id, row.query

    t0 = time.perf_counter()

    # ----- Full-catalog modality scores -----
    bm25_raw = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)
    sbert_raw = sbert_product_embeddings @ sbert_query_embeddings[qi]

    clip_raw_full = np.full(len(products), np.nan, dtype=np.float32)
    clip_raw_available = clip_image_embeddings @ clip_query_embeddings[qi]
    clip_raw_full[clip_catalog_indices] = clip_raw_available

    # ----- Independent candidate lists -----
    bm25_top100 = top_indices(bm25_raw, CANDIDATE_DEPTH)
    sbert_top100 = top_indices(sbert_raw, CANDIDATE_DEPTH)

    clip_local_top100 = top_indices(clip_raw_available, min(CANDIDATE_DEPTH, len(clip_raw_available)))
    clip_top100 = clip_catalog_indices[clip_local_top100]

    bm25_rank = rank_map(bm25_top100)
    sbert_rank = rank_map(sbert_top100)
    clip_rank = rank_map(clip_top100)

    # Candidate unions.
    union_bs = set(bm25_top100) | set(sbert_top100)
    union_sc = set(sbert_top100) | set(clip_top100)
    union_all = set(bm25_top100) | set(sbert_top100) | set(clip_top100)

    # RRF helper: missing list => zero contribution, never a negative penalty.
    def rrf_score(idx, use_bm25=True, use_sbert=True, use_clip=True):
        score = 0.0
        if use_bm25 and idx in bm25_rank:
            score += RRF_W_BM25 / (RRF_K + bm25_rank[idx])
        if use_sbert and idx in sbert_rank:
            score += RRF_W_SBERT / (RRF_K + sbert_rank[idx])
        if use_clip and idx in clip_rank:
            score += RRF_W_CLIP / (RRF_K + clip_rank[idx])
        return score

    method_scores = {}

    # 1) BM25 + SBERT RRF
    score = np.full(len(products), -1e9, dtype=np.float32)
    for idx in union_bs:
        score[idx] = rrf_score(idx, True, True, False)
    method_scores["BM25+SBERT_RRF"] = score

    # 2) SBERT + CLIP RRF
    score = np.full(len(products), -1e9, dtype=np.float32)
    for idx in union_sc:
        score[idx] = rrf_score(idx, False, True, True)
    method_scores["SBERT+CLIP_RRF"] = score

    # 3) BM25 + SBERT + CLIP RRF
    score = np.full(len(products), -1e9, dtype=np.float32)
    for idx in union_all:
        score[idx] = rrf_score(idx, True, True, True)
    method_scores["BM25+SBERT+CLIP_RRF"] = score

    # 4) Coverage-aware continuous score fusion.
    # Important: no-image products do NOT receive an image penalty.
    bm25_norm = minmax(bm25_raw)
    sbert_norm = minmax(sbert_raw)

    clip_norm_full = np.zeros(len(products), dtype=np.float32)
    clip_norm_available = minmax(clip_raw_available)
    clip_norm_full[clip_catalog_indices] = clip_norm_available

    has_image = np.zeros(len(products), dtype=np.float32)
    has_image[clip_catalog_indices] = 1.0

    numerator = (
        SCORE_W_BM25 * bm25_norm +
        SCORE_W_SBERT * sbert_norm +
        SCORE_W_CLIP * clip_norm_full * has_image
    )

    denominator = (
        SCORE_W_BM25 +
        SCORE_W_SBERT +
        SCORE_W_CLIP * has_image
    )

    coverage_score = numerator / denominator
    method_scores["CoverageAware_ScoreFusion"] = coverage_score

    # Candidate-generation diagnostic: union of top100 from all modalities.
    union_ranked = sorted(
        union_all,
        key=lambda idx: rrf_score(idx, True, True, True),
        reverse=True
    )

    positives = known_positives.get(qid, set())
    union_pids = [str(idx_to_pid[idx]) for idx in union_ranked]

    union_recall_100 = recall_at_k(union_pids, positives, min(100, len(union_pids)))
    union_recall_300 = recall_at_k(union_pids, positives, min(300, len(union_pids)))

    # Evaluate each final method.
    judged_pids = gold.loc[gold["query_id"] == qid, "product_id"].tolist()

    for method, scores in method_scores.items():
        final_idx = top_indices(scores, TOP_K)
        retrieved_pids = [str(idx_to_pid[idx]) for idx in final_idx]

        score_by_pid = {
            pid: float(scores[pid_to_idx[pid]])
            for pid in judged_pids if pid in pid_to_idx
        }

        ndcg, mrr = judged_metrics(qid, score_by_pid, 10)

        metric_rows.append({
            "method": method,
            "query_id": qid,
            "query": query,
            "NDCG@10_judged": ndcg,
            "MRR_judged": mrr,
            "Recall@10_full": recall_at_k(retrieved_pids, positives, 10),
            "Recall@50_full": recall_at_k(retrieved_pids, positives, 50),
            "CandidateRecall@100": union_recall_100,
            "CandidateRecall@300": union_recall_300,
        })

        for rank, idx in enumerate(final_idx, start=1):
            pid = str(idx_to_pid[idx])
            top50_rows.append({
                "method": method,
                "query_id": qid,
                "query": query,
                "rank": rank,
                "product_id": pid,
                "relevance_label": relevance_lookup[qid].get(pid, np.nan),
                "score": float(scores[idx]),
                "has_image": bool(idx in set(clip_catalog_indices)),
            })

    latency_rows.append({
        "query_id": qid,
        "latency_ms_all_v3_methods": (time.perf_counter()-t0)*1000
    })

    if (qi+1) % 25 == 0 or (qi+1) == len(test_queries):
        print(f"Processed {qi+1}/{len(test_queries)} queries")

v3_metrics_per_query = pd.DataFrame(metric_rows)
v3_top50 = pd.DataFrame(top50_rows)
v3_latency = pd.DataFrame(latency_rows)

print("V3 retrieval complete.")

In [ ]:
# ============================================================
# 10. SUMMARY + ROADMAP GATE
# ============================================================

summary_rows = []

for method, g in v3_metrics_per_query.groupby("method"):
    summary_rows.append({"model": method, **summarize(g)})

v3_summary = pd.DataFrame(summary_rows)

v1_metrics = pd.read_csv(V1_METRICS_PATH)
v2_metrics = pd.read_csv(V2_METRICS_PATH)

baseline_cols = [
    "model","NDCG@10_judged","MRR_judged",
    "Recall@10_full","Recall@50_full","queries_evaluated"
]

baseline = pd.concat([
    v1_metrics[baseline_cols],
    v2_metrics[
        v2_metrics["model"].isin(["CLIP-image","SBERT+CLIP"])
    ][baseline_cols]
], ignore_index=True).drop_duplicates("model")

baseline["CandidateRecall@100"] = np.nan
baseline["CandidateRecall@300"] = np.nan

final_comparison = pd.concat([
    baseline,
    v3_summary
], ignore_index=True)

sbert_recall50 = final_comparison.loc[
    final_comparison["model"] == "SBERT+FAISS",
    "Recall@50_full"
].iloc[0]

target = sbert_recall50 * 1.10

final_comparison["Recall@50_vs_SBERT_%"] = (
    (final_comparison["Recall@50_full"] - sbert_recall50)
    / sbert_recall50 * 100
)

final_comparison["Meets_10pct_multimodal_gate"] = (
    final_comparison["Recall@50_full"] >= target
)

print("SBERT text-only Recall@50:", round(float(sbert_recall50), 4))
print("Week 4 +10% target:", round(float(target), 4))
display(final_comparison.round(4))

In [ ]:
# ============================================================
# 11. PER-QUERY DIAGNOSTICS
# ============================================================

pivot = v3_metrics_per_query.pivot_table(
    index=["query_id","query"],
    columns="method",
    values=["NDCG@10_judged","Recall@50_full"]
)

pivot.columns = [
    f"{metric}__{method}"
    for metric, method in pivot.columns
]
pivot = pivot.reset_index()

# Compare the full hybrid RRF against SBERT baseline from V1.
v1_sbert_path = list(EXTRACT_DIR.rglob("week4_sbert_query_metrics.csv"))
if v1_sbert_path:
    sbert_q = pd.read_csv(v1_sbert_path[0])
    sbert_q["query_id"] = sbert_q["query_id"].astype(str)

    pivot = pivot.merge(
        sbert_q[["query_id","NDCG@10_judged","Recall@50_full"]],
        on="query_id",
        how="left"
    ).rename(columns={
        "NDCG@10_judged":"NDCG@10_judged__SBERT_BASELINE",
        "Recall@50_full":"Recall@50_full__SBERT_BASELINE"
    })

    hybrid_col = "Recall@50_full__BM25+SBERT+CLIP_RRF"
    pivot["Recall50_Hybrid_minus_SBERT"] = (
        pivot[hybrid_col] - pivot["Recall@50_full__SBERT_BASELINE"]
    )

    print("Largest Recall@50 gains from hybrid RRF:")
    display(
        pivot.sort_values("Recall50_Hybrid_minus_SBERT", ascending=False)
        .head(15)[[
            "query_id","query",
            "Recall@50_full__SBERT_BASELINE",
            hybrid_col,
            "Recall50_Hybrid_minus_SBERT"
        ]]
    )

    print("\nLargest Recall@50 losses from hybrid RRF:")
    display(
        pivot.sort_values("Recall50_Hybrid_minus_SBERT")
        .head(15)[[
            "query_id","query",
            "Recall@50_full__SBERT_BASELINE",
            hybrid_col,
            "Recall50_Hybrid_minus_SBERT"
        ]]
    )

In [ ]:
# ============================================================
# 12. LATENCY + 10K RELIABILITY CHECK
# ============================================================

lat_summary = pd.DataFrame([{
    "component": "All four V3 hybrid methods per query",
    "n_queries": len(v3_latency),
    "mean_ms": v3_latency["latency_ms_all_v3_methods"].mean(),
    "median_ms": v3_latency["latency_ms_all_v3_methods"].median(),
    "p95_ms": v3_latency["latency_ms_all_v3_methods"].quantile(0.95),
    "max_ms": v3_latency["latency_ms_all_v3_methods"].max(),
}])

display(lat_summary.round(3))

print(
    "10k-catalog V3 P95 < 50 ms:",
    "PASS" if lat_summary["p95_ms"].iloc[0] < 50 else "NOT YET",
    f'({lat_summary["p95_ms"].iloc[0]:.3f} ms)'
)

print("\nNote:")
print("This latency includes computing all 4 experimental V3 methods.")
print("A production system would normally run only the selected retrieval architecture.")

In [ ]:
# ============================================================
# 13. PLOTS
# ============================================================

plot_df = final_comparison.set_index("model")[
    ["NDCG@10_judged","MRR_judged","Recall@10_full","Recall@50_full"]
]

ax = plot_df.T.plot(kind="bar", figsize=(13,6))
ax.set_title("Week 4 — Retrieval Architecture Comparison")
ax.set_ylabel("Metric")
ax.set_ylim(0,1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "week4_v3_retrieval_comparison.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

v3_only = v3_summary.set_index("model")[
    ["CandidateRecall@100","CandidateRecall@300"]
]

ax = v3_only.plot(kind="bar", figsize=(10,5))
ax.set_title("Week 4 V3 — Hybrid Candidate-Union Recall")
ax.set_ylabel("Recall")
ax.set_ylim(0,1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "week4_v3_candidate_union_recall.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 14. EXPORT
# ============================================================

v3_metrics_per_query.to_csv(
    OUTPUT_DIR / "week4_v3_query_metrics.csv",
    index=False
)

v3_top50.to_csv(
    OUTPUT_DIR / "week4_v3_top50_all_methods.csv",
    index=False
)

v3_summary.to_csv(
    OUTPUT_DIR / "week4_v3_summary.csv",
    index=False
)

final_comparison.to_csv(
    OUTPUT_DIR / "week4_v3_final_comparison.csv",
    index=False
)

v3_latency.to_csv(
    OUTPUT_DIR / "week4_v3_latency_per_query.csv",
    index=False
)

lat_summary.to_csv(
    OUTPUT_DIR / "week4_v3_latency_summary.csv",
    index=False
)

pivot.to_csv(
    OUTPUT_DIR / "week4_v3_per_query_diagnostics.csv",
    index=False
)

image_coverage = pd.read_csv(IMAGE_COVERAGE_PATH)
image_coverage.to_csv(
    OUTPUT_DIR / "week4_v3_image_coverage_reference.csv",
    index=False
)

readme = f'''
Amazon Search / LTR — Week 4 V3
================================

Purpose
-------
Fix the partial-image-coverage failure mode from V2 without entering Week 5.

Methods
-------
1. BM25 + SBERT reciprocal-rank fusion
2. SBERT + CLIP reciprocal-rank fusion
3. BM25 + SBERT + CLIP reciprocal-rank fusion
4. Coverage-aware continuous score fusion

RRF
---
Candidate depth per modality: {CANDIDATE_DEPTH}
RRF k: {RRF_K}
BM25 weight: {RRF_W_BM25}
SBERT weight: {RRF_W_SBERT}
CLIP weight: {RRF_W_CLIP}

Coverage-aware score fusion
---------------------------
BM25 weight: {SCORE_W_BM25}
SBERT weight: {SCORE_W_SBERT}
CLIP weight: {SCORE_W_CLIP}

If a product has no image, the CLIP term is omitted and the remaining
weights are renormalized. Missing images therefore do not create an
automatic score penalty.

Important
---------
This is still candidate retrieval / multimodal retrieval.
No LambdaMART, LightGBM, pairwise/listwise ranking, or Week 5
feature-learning pipeline is used.
'''

with open(OUTPUT_DIR / "README_WEEK4_V3.txt","w",encoding="utf-8") as f:
    f.write(readme)

zip_path = shutil.make_archive(
    str(ROOT / "amazon_week4_v3_coverage_aware_hybrid_results"),
    "zip",
    OUTPUT_DIR
)

print("Created:", zip_path)

In [ ]:
# ============================================================
# 15. FINAL SCORECARD + DOWNLOAD
# ============================================================

print("="*90)
print("WEEK 4 V3 COMPLETE")
print("="*90)

display(final_comparison.round(4))
display(lat_summary.round(3))

print("\nWeek 4 multimodal target:")
print("Required Recall@50 >=", round(float(target), 4))

for _, r in final_comparison.iterrows():
    if r["model"] in [
        "SBERT+CLIP_RRF",
        "BM25+SBERT+CLIP_RRF",
        "CoverageAware_ScoreFusion"
    ]:
        status = "PASS" if bool(r["Meets_10pct_multimodal_gate"]) else "NOT YET"
        print(
            f'{r["model"]}: Recall@50={r["Recall@50_full"]:.4f} '
            f'({r["Recall@50_vs_SBERT_%"]:+.2f}% vs SBERT) -> {status}'
        )

print("\nNo Week 5 model was trained.")

if is_colab:
    files.download(
        str(ROOT / "amazon_week4_v3_coverage_aware_hybrid_results.zip")
    )
else:
    print(ROOT / "amazon_week4_v3_coverage_aware_hybrid_results.zip")